# mt_evaluation — Shared Evaluation Metrics

Centralised metric functions used by all four architectures. Loaded once by `00_main`.

## Primary Metrics (Claim-Level)

| Metric | Field | Method |
|---|---|---|
| **Answer Precision** | `answer_precision` | Generated claims entailed by ground truth / total generated claims |
| **Answer Recall** | `answer_recall` | Expected claims covered by generated answer / total expected claims |
| **Answer F1** | `answer_f1` | Harmonic mean of precision and recall |
| **Claim Groundedness** | `claim_groundedness` | Generated claims supported by evidence / total factual claims |

## Supporting Metrics

| Metric | Field |
|---|---|
| Numeric Hallucination Risk | `numeric_hallucination_risk` |
| SQL Execution Status | `sql_execution_status` |
| SQL Correctness Score | `sql_correctness_score` |
| Abstention Correctness | `abstention_correctness` |
| Workflow Success | `workflow_success` |

In [0]:
import re

# ─────────────────────────────────────────────────────────────────────────────
# Shared tolerance and helpers
# ─────────────────────────────────────────────────────────────────────────────

NUMERIC_TOLERANCE = 0.05   # 5% relative tolerance for numeric comparisons


def _extract_numbers(text: str) -> list:
    """Extract all numeric values from text (handles commas and percentages)."""
    nums = []
    for n in re.findall(r'\b[\d,]+\.?\d*\b', str(text)):
        try:
            nums.append(float(n.replace(',', '')))
        except ValueError:
            pass
    return nums


def _extract_words(text: str, min_len: int = 5) -> list:
    """Extract lowercase alphabetic words of at least min_len characters."""
    return [w.lower() for w in re.findall(rf'\b[a-zA-Z]{{{min_len},}}\b', str(text))]


def _num_match(a: float, b: float, tol: float = NUMERIC_TOLERANCE) -> bool:
    """True if two numbers are within relative tolerance."""
    if b == 0:
        return a == 0
    return abs(a - b) / abs(b) <= tol


# ─────────────────────────────────────────────────────────────────────────────
# 1. Answer Correctness Proxy
# ─────────────────────────────────────────────────────────────────────────────

def compute_answer_correctness_proxy(generated: str, expected: str) -> float:
    """
    Hybrid numeric + keyword correctness score.

    Numeric check (60% weight):
      Extracts numbers from the expected answer and checks if equivalent values
      appear in the generated answer within 5% relative tolerance.

    Keyword check (40% weight):
      Word overlap for alphabetic tokens of 4+ characters.

    Returns:
      1.0  = correct   (combined score >= 0.65)
      0.5  = partially correct (combined score >= 0.30)
      0.0  = incorrect
      None = not applicable (missing input)
    """
    if not generated or not expected:
        return None

    exp_nums = _extract_numbers(expected)
    gen_nums = _extract_numbers(generated)

    if exp_nums:
        hits = sum(1 for en in exp_nums if any(_num_match(en, gn) for gn in gen_nums))
        num_score = hits / len(exp_nums)
    else:
        num_score = None

    exp_words = re.findall(r'\b[a-zA-Z]{4,}\b', expected.lower())
    gen_lower  = generated.lower()
    if exp_words:
        kw_score = sum(1 for w in exp_words if w in gen_lower) / len(exp_words)
    else:
        kw_score = None

    if num_score is not None and kw_score is not None:
        combined = 0.6 * num_score + 0.4 * kw_score
    elif num_score is not None:
        combined = num_score
    elif kw_score is not None:
        combined = kw_score
    else:
        return 0.0

    if combined >= 0.65:
        return 1.0
    elif combined >= 0.30:
        return 0.5
    else:
        return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 2. Groundedness Score
# ─────────────────────────────────────────────────────────────────────────────

def compute_groundedness_score(answer: str, evidence: str) -> tuple:
    """
    Dynamic groundedness: checks if the answer is supported by the actual
    evidence from the current run (SQL results string or retrieved chunks).
    No hardcoded keyword lists — uses the evidence from this specific run.

    Numeric grounding (60%):
      Numbers in the answer that appear in evidence (5% tolerance).
    Keyword grounding (40%):
      Domain words in the answer that appear in evidence (capped at 60 words).

    Returns: (score: float, explanation: str)
      score = 0.0–1.0, or None if not applicable.
    """
    if not answer:
        return (None, "No answer to evaluate")
    if not evidence or str(evidence).strip() in ("", "None", "[]", "N/A",
                                                   "[SQL execution not available outside Databricks]"):
        return (None, "No evidence available for grounding check")

    ans_nums = _extract_numbers(answer)
    ev_nums  = _extract_numbers(evidence)

    if ans_nums and ev_nums:
        supported = sum(1 for an in ans_nums if any(_num_match(an, en) for en in ev_nums))
        num_ratio = supported / len(ans_nums)
        num_note  = f"{supported}/{len(ans_nums)} numbers supported"
    elif ans_nums:
        num_ratio = 0.0
        num_note  = f"{len(ans_nums)} numbers in answer, none traceable to evidence"
    else:
        num_ratio = 1.0
        num_note  = "no numeric claims"

    ans_words = _extract_words(answer, min_len=5)[:60]
    ev_set    = set(_extract_words(evidence, min_len=5))
    if ans_words:
        kw_hit   = sum(1 for w in ans_words if w in ev_set)
        kw_ratio = kw_hit / len(ans_words)
        kw_note  = f"{kw_hit}/{len(ans_words)} key terms supported"
    else:
        kw_ratio = 1.0
        kw_note  = "no long-word claims"

    score       = round(min(0.6 * num_ratio + 0.4 * kw_ratio, 1.0), 2)
    explanation = f"Numeric: {num_note} | Keyword: {kw_note}"
    return (score, explanation)


# ─────────────────────────────────────────────────────────────────────────────
# 3. Numeric Hallucination Risk
# ─────────────────────────────────────────────────────────────────────────────

def compute_numeric_hallucination_risk(
    answer: str,
    sql_results: str,
    tolerance: float = NUMERIC_TOLERANCE
) -> tuple:
    """
    Detects numeric values in the answer that are NOT traceable to the SQL results.
    Applies rounding tolerance to avoid penalising legitimately rounded figures
    (e.g. 68.3% rounded to 68% is not considered a hallucination).

    Returns: (risk_score: float, hallucination_flag: bool)
      risk_score = fraction of unsupported numbers [0.0–1.0]
      hallucination_flag = True if risk_score > 0.30
    """
    if not answer:
        return (None, None)

    ans_nums = _extract_numbers(answer)
    if not ans_nums:
        return (0.0, False)

    ctx_nums = _extract_numbers(str(sql_results)) if sql_results else []

    if not ctx_nums:
        return (1.0, True)   # no context to verify against

    unsupported = [
        an for an in ans_nums
        if not any(_num_match(an, cn, tolerance) for cn in ctx_nums)
    ]
    risk = round(len(unsupported) / len(ans_nums), 2)
    flag = risk > 0.30
    return (risk, flag)


# ─────────────────────────────────────────────────────────────────────────────
# 4. SQL Correctness Heuristic
# ─────────────────────────────────────────────────────────────────────────────

def compute_sql_correctness_heuristic(
    generated_sql: str,
    tables_needed: list,
    sql_execution_status: str
) -> float:
    """
    Heuristic SQL correctness. Primary signal: whether the query references
    the correct tables (as declared in tables_needed for the question).
    Secondary signal: presence of aggregation and filter/grouping clauses.

    NOTE: this is a proxy — it cannot verify join logic, business-rule
    correctness, or full semantic alignment. Use sql_correctness_score
    as a diagnostic signal and not as a definitive correctness measure.

    Returns:
      1.0  = correct tables + aggregation present
      0.5  = partial table match or structural issues
      0.0  = wrong tables or SQL execution failed
      None = no SQL generated / not applicable
    """
    if not generated_sql:
        return None
    if sql_execution_status == "not_applicable":
        return None
    if sql_execution_status == "failed":
        return 0.0

    sql_lower = generated_sql.lower()
    tables    = tables_needed or []

    tables_hit  = sum(1 for t in tables if t.lower() in sql_lower)
    table_ratio = tables_hit / len(tables) if tables else 0.5

    has_aggregation  = bool(re.search(r'\b(count|sum|avg|max|min|measure)\b', sql_lower))
    has_group_filter = bool(re.search(r'\b(group\s+by|where)\b', sql_lower))

    if table_ratio == 1.0 and has_aggregation:
        return 1.0
    elif table_ratio >= 0.5 or (table_ratio == 0.0 and has_aggregation and has_group_filter):
        return 0.5
    else:
        return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 5. Semantic Alignment Heuristic
# ─────────────────────────────────────────────────────────────────────────────

def compute_semantic_alignment_heuristic(
    generated_sql: str,
    mode: str,
    semantic_context: str = None
) -> float:
    """
    Heuristic for whether the SQL follows semantic layer definitions.
    Only applicable to SEMANTIC modes; returns None for RAW mode.

    Checks: whether the SQL references the expected metric views / semantic
    tables or uses MEASURE() syntax and domain-specific terminology.

    Returns: 1.0 / 0.5 / 0.0 / None (not applicable)
    """
    if not mode or "SEMANTIC" not in mode.upper():
        return None
    if not generated_sql:
        return None

    sql_lower = generated_sql.lower()

    # Semantic table/view signals expected in this project
    TABLE_SIGNALS = [
        "rc_certificate_metrics",
        "rc_compliance_metrics",
        "rc_revenue_metrics",
        "mt_safe_certificate",
        "mt_safe_assignment_course_progress",
        "measure(",
    ]
    KEYWORD_SIGNALS = [
        "certificate count",
        "distinct users",
        "on-time completion",
        "on time count",
        "pflichtkurs",
        "wahlkurs",
        "year_month",
        "company_id2",
    ]

    table_hits = sum(1 for s in TABLE_SIGNALS   if s in sql_lower)
    kw_hits    = sum(1 for s in KEYWORD_SIGNALS if s in sql_lower)

    if table_hits >= 1 or kw_hits >= 2:
        return 1.0
    elif kw_hits == 1:
        return 0.5
    else:
        return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 6. Abstention Correctness
# ─────────────────────────────────────────────────────────────────────────────

def compute_abstention_correctness(answer: str, answerability_label: str) -> bool:
    """
    Checks if the agent correctly refused to answer an unanswerable question.
    Returns None if the question IS answerable (metric not applicable).

    Returns:
      True  = agent correctly refused
      False = agent fabricated an answer
      None  = question is answerable, metric not applicable
    """
    if not answerability_label or answerability_label.lower() in ("answerable", "yes", ""):
        return None

    REFUSAL_PHRASES = [
        "cannot answer", "not available", "insufficient data",
        "not enough information", "cannot be determined",
        "data not available", "unable to answer", "not found",
        "no information", "cannot be answered",
        "i don't have", "i do not have",
    ]
    if not answer:
        return True  # silence = correct abstention
    return any(p in answer.lower() for p in REFUSAL_PHRASES)


# ─────────────────────────────────────────────────────────────────────────────
# 7. SQL Execution Status Helper
# ─────────────────────────────────────────────────────────────────────────────

def infer_sql_execution_status(sql_results: str, generated_sql: str) -> tuple:
    """
    Infers sql_execution_status and sql_error_message from the sql_results string
    returned by execute_sql_query(), without modifying the core agent function.

    Returns: (status: str, error_message: str | None)
      status = 'success' | 'failed' | 'not_applicable'
    """
    if not generated_sql:
        return ("not_applicable", None)
    if not sql_results:
        return ("not_applicable", None)
    r = str(sql_results)
    if r.startswith("SQL ERROR:"):
        return ("failed", r.replace("SQL ERROR:", "").strip())
    if "[SQL execution not available" in r or "[mock]" in r.lower():
        return ("not_applicable", None)
    return ("success", None)


# ─────────────────────────────────────────────────────────────────────────────
# 8. Extract Table Names from SQL
# ─────────────────────────────────────────────────────────────────────────────

def extract_tables_from_sql(sql: str) -> list:
    """
    Extract table names referenced in a SQL query (FROM / JOIN clauses).
    Returns a deduplicated list of lowercase table name strings.
    """
    if not sql:
        return []
    pattern = r'(?:FROM|JOIN)\s+([\w.`]+)'
    matches = re.findall(pattern, sql, re.IGNORECASE)
    return list(set(m.strip('`').lower() for m in matches))


print("✓ mt_evaluation helper functions loaded")

In [0]:
# =============================================================================
# CLAIM-LEVEL EVALUATION (RAGChecker-inspired)
# Ru et al. (2024), "RAGChecker", NeurIPS 2024
# =============================================================================
import hashlib
import json
import unicodedata
from typing import Optional

# --- Extraction cache (in-memory per session) ---
_claim_extraction_cache = {}


def _cache_key(question: str, answer: str, model: str, prompt_version: str) -> str:
    """SHA-256 hash for claim extraction cache."""
    payload = f"{question}|{answer}|{model}|{prompt_version}"
    return hashlib.sha256(payload.encode()).hexdigest()


# --- Normalisation utilities ---

def normalise_text(text: str) -> str:
    """Whitespace, case, Unicode normalisation."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalise_numeric_string(val_str: str) -> Optional[float]:
    """Parse numeric values handling US/EU formats and suffixes."""
    if not val_str:
        return None
    s = str(val_str).strip().lower()
    # Remove currency symbols
    s = re.sub(r'[\$\u20ac\xa3]', '', s)
    # Handle percentage → keep as-is (e.g. 68% → 68.0)
    s = s.rstrip('%')
    # Handle k/m/b suffixes
    multiplier = 1.0
    if s.endswith('k'):
        s = s[:-1]; multiplier = 1_000
    elif s.endswith('m'):
        s = s[:-1]; multiplier = 1_000_000
    elif s.endswith('b'):
        s = s[:-1]; multiplier = 1_000_000_000
    # Detect EU format: "1.234,56" → has dot before comma
    if re.match(r'^\d{1,3}(\.\d{3})+(,\d+)?$', s):
        s = s.replace('.', '').replace(',', '.')
    # US format: "1,234.56" → remove commas
    elif ',' in s:
        s = s.replace(',', '')
    try:
        return float(s) * multiplier
    except ValueError:
        return None


def normalise_alias(value: str, alias_dict: dict) -> str:
    """Resolve entity/metric aliases to canonical form."""
    if not value:
        return value
    v_lower = value.lower().strip()
    for canonical, aliases in alias_dict.items():
        if v_lower == canonical or v_lower in aliases:
            return canonical
    return v_lower


# --- Claim extraction prompt ---

_CLAIM_EXTRACTION_PROMPT = """You are a claim extractor. Given a question and an answer, extract ALL atomic factual claims from the answer.

Rules:
- Each claim must be ONE independently verifiable fact
- Split conjunctions containing multiple facts into separate claims
- Preserve: entity, metric, number, unit, period, conditions, negations
- Resolve pronouns using the question and answer context
- Do NOT infer or add facts not stated in the answer
- Remove greetings, opinions, and non-factual filler
- Deduplicate equivalent claims
- Do NOT treat list numbering (1, 2, 3...) as numeric values
- IMPORTANT: The answer may be written as an audit report, validation analysis,
  or quality review rather than a plain business answer. Extract claims from the
  DATA, NUMBERS, and FACTUAL STATEMENTS regardless of the answer's framing.
  Look inside markdown tables, bullet lists, and embedded SQL results for facts.

Return ONLY a JSON array. Each element must have this schema:
{{
  "claim_id": "<source>_C<N>",
  "text": "self-contained atomic factual statement",
  "entity": "string or null",
  "metric": "string or null",
  "value": number or string or null,
  "unit": "string or null",
  "period": "string or null",
  "qualifiers": {{}},
  "required": true
}}

Question: {question}
Answer: {answer}
Source type: {source_type}

JSON array of claims:"""


def _validate_claim_schema(claim: dict) -> bool:
    """Validate a single claim dict has required fields."""
    required_fields = {"claim_id", "text"}
    return required_fields.issubset(claim.keys()) and isinstance(claim.get("text"), str)


def extract_atomic_claims(
    question: str,
    answer: str,
    source_type: str = "generated",
    params: dict = None
) -> list:
    """
    Extract atomic claims from an answer using the fixed evaluator model.
    Uses caching and retries. Returns list of claim dicts or evaluator_error marker.
    """
    if not answer or not answer.strip():
        return []

    params = params or CLAIM_EVALUATION_PARAMS
    model = params["extractor_model"]
    prompt_version = params["prompt_version"]
    max_retries = params["max_retries"]

    # Check cache
    cache_k = _cache_key(question, answer, model, prompt_version)
    if cache_k in _claim_extraction_cache:
        return _claim_extraction_cache[cache_k]

    # Build prompt
    prompt = _CLAIM_EXTRACTION_PROMPT.format(
        question=question, answer=answer, source_type=source_type
    )

    # LLM client — reuse the shared 429-resilient client if available
    eval_client = LLM_CLIENT if 'LLM_CLIENT' in dir() else None
    if eval_client is None:
        from openai import OpenAI
        workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
        _token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        eval_client = OpenAI(api_key=_token, base_url=f"https://{workspace_url}/serving-endpoints")

    for attempt in range(max_retries + 1):
        try:
            resp = eval_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=params["temperature"],
                max_tokens=params["max_tokens"]
            )
            raw_text = resp.choices[0].message.content.strip()

            # Extract JSON array from response
            json_match = re.search(r'\[.*\]', raw_text, re.DOTALL)
            if not json_match:
                if attempt < max_retries:
                    continue
                return [{"_evaluator_error": True, "error": "No JSON array in response", "raw": raw_text[:500]}]

            claims = json.loads(json_match.group())
            if not isinstance(claims, list):
                if attempt < max_retries:
                    continue
                return [{"_evaluator_error": True, "error": "Response is not a list"}]

            # Validate schema and assign IDs
            valid_claims = []
            for i, c in enumerate(claims):
                if not isinstance(c, dict):
                    continue
                # Ensure required fields
                c.setdefault("claim_id", f"{source_type}_C{i+1}")
                c.setdefault("text", "")
                c.setdefault("entity", None)
                c.setdefault("metric", None)
                c.setdefault("value", None)
                c.setdefault("unit", None)
                c.setdefault("period", None)
                c.setdefault("qualifiers", {})
                c.setdefault("required", True)
                if _validate_claim_schema(c) and c["text"].strip():
                    valid_claims.append(c)

            if valid_claims:
                _claim_extraction_cache[cache_k] = valid_claims
                return valid_claims
            elif attempt < max_retries:
                continue
            else:
                return [{"_evaluator_error": True, "error": "No valid claims extracted"}]

        except json.JSONDecodeError as e:
            if attempt >= max_retries:
                return [{"_evaluator_error": True, "error": f"JSON parse error: {e}"}]
        except Exception as e:
            if attempt >= max_retries:
                return [{"_evaluator_error": True, "error": f"Extraction failed: {e}"}]

    return [{"_evaluator_error": True, "error": "Max retries exceeded"}]


def prepare_expected_claims(questions: list, force: bool = False) -> list:
    """
    Extract expected_claims from expected_answer for questions that lack them.
    Persists results into the question dicts (call once, then review).
    """
    updated = 0
    for q in questions:
        if not force and q.get("expected_claims"):
            continue
        claims = extract_atomic_claims(
            question=q["question"],
            answer=q["expected_answer"],
            source_type="expected"
        )
        if claims and not any(c.get("_evaluator_error") for c in claims):
            q["expected_claims"] = claims
            updated += 1
    return updated


print("✓ Claim extraction & normalisation loaded")

In [0]:
# =============================================================================
# NUMERIC CLAIM MATCHING & SEMANTIC ENTAILMENT
# =============================================================================

def _stem(word: str) -> str:
    """Minimal stemmer: strips common English plural/verb suffixes."""
    w = word.lower()
    if w.endswith('ies') and len(w) > 4:
        return w[:-3] + 'y'  # companies → company
    if w.endswith('sses'):
        return w[:-2]  # processes → process
    if w.endswith('ches') or w.endswith('shes'):
        return w[:-2]  # batches → batch
    if w.endswith('xes') or w.endswith('zes'):
        return w[:-2]  # taxes → tax
    if w.endswith('ses') and len(w) > 4:
        base_s = w[:-1]
        if base_s.endswith(('se', 'ce')):
            return base_s  # courses → course, houses → house
        return w[:-2]
    if w.endswith('es') and len(w) > 4:
        return w[:-1]  # rates → rate, certificates → certificate
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3:
        return w[:-1]  # counts → count
    return w


def _metric_compatible(g_metric: str, e_metric: str) -> bool:
    """
    Check if two metric names are semantically compatible.
    Uses stemmed word overlap: if metrics share at least one significant
    stemmed word, they are compatible. This handles cases like
    'company count' vs 'active companies' or 'certificate count' vs 'total certificates'.
    """
    if not g_metric or not e_metric:
        return True  # Missing metric = no constraint
    if g_metric == e_metric:
        return True
    # Stemmed word overlap check
    stop = {"of", "the", "a", "an", "per", "for", "in", "by", "with", "and", "or", "is", "are", "avg", "average"}
    g_words = {_stem(w) for w in g_metric.lower().split()} - stop
    e_words = {_stem(w) for w in e_metric.lower().split()} - stop
    if g_words & e_words:
        return True  # At least one shared stemmed word
    # Check if one contains the other as substring
    if g_metric in e_metric or e_metric in g_metric:
        return True
    return False


def _entity_compatible(g_entity: str, e_entity: str) -> bool:
    """
    Check if two entity names are compatible.
    Handles pseudonymised IDs (C_xxx) and domain terms (baercare, care).
    """
    if not g_entity or not e_entity:
        return True  # Missing entity = no constraint
    if g_entity == e_entity:
        return True
    # Check if one contains the other
    if g_entity in e_entity or e_entity in g_entity:
        return True
    return False


def _period_compatible(g_period: str, e_period: str) -> bool:
    """
    Check if two period strings are compatible.
    Handles 'january 2026' vs 'jan 2026', '2026' vs 'through july 2026', etc.
    """
    if not g_period or not e_period:
        return True  # Missing period = no constraint
    if g_period == e_period:
        return True
    # Check containment
    if g_period in e_period or e_period in g_period:
        return True
    # Check if they share the same year
    g_years = re.findall(r'20\d{2}', g_period)
    e_years = re.findall(r'20\d{2}', e_period)
    if g_years and e_years and g_years[0] == e_years[0]:
        # Same year — check if months match (if mentioned)
        month_map = {"jan": "01", "feb": "02", "mar": "03", "apr": "04",
                     "may": "05", "jun": "06", "jul": "07", "aug": "08",
                     "sep": "09", "oct": "10", "nov": "11", "dec": "12"}
        g_months = [m for m in month_map if m in g_period]
        e_months = [m for m in month_map if m in e_period]
        if not g_months or not e_months:
            return True  # One side has no month = compatible
        if set(g_months) & set(e_months):
            return True  # At least one shared month
    return False


def numeric_claim_match(
    generated_claim: dict,
    expected_claim: dict,
    tolerance_config: dict = None
) -> bool:
    """
    Match a generated claim against an expected claim using VALUE-FIRST logic.
    
    Strategy: if the numeric values match within tolerance, accept the match
    UNLESS entity/metric/period fields explicitly CONTRADICT each other.
    This fixes the precision bug where correct values were missed because
    the LLM claim extractor used different metric terminology than expected.
    
    A claim matches when:
    1. Values match within tolerance, AND
    2. Entity fields are compatible (not contradictory), AND
    3. Metric fields are compatible (share word overlap), AND
    4. Period fields are compatible (share year/month)
    """
    tc = tolerance_config or CLAIM_EVALUATION_PARAMS
    rel_tol = tc.get("default_relative_tolerance", 0.01)
    abs_tol = tc.get("default_absolute_tolerance", 1e-6)
    pct_tol = tc.get("percentage_point_tolerance", 0.5)

    g_val = generated_claim.get("value")
    e_val = expected_claim.get("value")

    # Both must have values for numeric comparison
    if g_val is None or e_val is None:
        return False

    # Parse to float if strings
    g_num = normalise_numeric_string(str(g_val)) if not isinstance(g_val, (int, float)) else float(g_val)
    e_num = normalise_numeric_string(str(e_val)) if not isinstance(e_val, (int, float)) else float(e_val)

    if g_num is None or e_num is None:
        # Exact string match fallback for non-numeric values (dates, IDs, categories)
        return normalise_text(str(g_val)) == normalise_text(str(e_val))

    # --- Decimal-to-percent normalisation ---
    # Agents often report rates as 0.56 (decimal) while expected claims use 55.7 (%).
    # Detect the 0–1 vs 0–100 scale mismatch and normalise before comparison.
    e_unit_norm = normalise_alias(expected_claim.get("unit"), aliases.get("units", {}) if 'aliases' in dir() else {})
    if e_unit_norm == "%" and 0 < abs(g_num) <= 1.0 and abs(e_num) > 1.0:
        g_num = g_num * 100.0  # 0.56 → 56.0
    elif e_unit_norm == "%" and abs(g_num) > 1.0 and 0 < abs(e_num) <= 1.0:
        e_num = e_num * 100.0  # reverse case

    # Resolve aliases
    aliases = CLAIM_ALIASES if 'CLAIM_ALIASES' in dir() else {"entities": {}, "metrics": {}, "units": {}}
    g_entity = normalise_alias(generated_claim.get("entity"), aliases.get("entities", {}))
    e_entity = normalise_alias(expected_claim.get("entity"), aliases.get("entities", {}))
    g_metric = normalise_alias(generated_claim.get("metric"), aliases.get("metrics", {}))
    e_metric = normalise_alias(expected_claim.get("metric"), aliases.get("metrics", {}))
    g_period = normalise_text(generated_claim.get("period") or "")
    e_period = normalise_text(expected_claim.get("period") or "")
    g_unit = normalise_alias(generated_claim.get("unit"), aliases.get("units", {}))
    e_unit = normalise_alias(expected_claim.get("unit"), aliases.get("units", {}))

    # Check compatibility (not strict equality)
    if not _entity_compatible(g_entity, e_entity):
        return False
    if not _metric_compatible(g_metric, e_metric):
        return False
    if not _period_compatible(g_period, e_period):
        return False

    # Numeric matching with appropriate tolerance
    is_count = any(kw in (g_metric or "") + (e_metric or "") for kw in ["count", "number", "total"])
    is_id = any(kw in (g_entity or "") for kw in ["id", "_id"])
    is_year = (g_period or "").isdigit() and len(g_period or "") == 4
    if is_id or is_year:
        return g_num == e_num  # IDs and years must be exact
    if is_count:
        # Allow ±2% tolerance for counts (different SQL approaches may count differently)
        count_tol = tc.get("count_relative_tolerance", 0.02)
        if e_num == 0:
            return g_num == 0
        return abs(g_num - e_num) / abs(e_num) <= count_tol

    # Percentage tolerance
    is_pct = (g_unit == "%" or e_unit == "%")
    if is_pct:
        return abs(g_num - e_num) <= pct_tol

    # General numeric tolerance
    return abs(g_num - e_num) <= max(abs_tol, rel_tol * abs(e_num))


# --- Entailment prompt ---
_ENTAILMENT_PROMPT = """You are a claim entailment checker. Determine if a CLAIM is entailed by, contradicted by, or not supported by the REFERENCE.

CLAIM: {claim_text}

REFERENCE:
{reference_text}

Classify the relationship:
- ENTAILED: The reference clearly supports or implies this claim
- CONTRADICTED: The reference states something incompatible with this claim
- NOT_ENOUGH_INFORMATION: The reference neither supports nor contradicts this claim

Return ONLY valid JSON:
{{"label": "ENTAILED|CONTRADICTED|NOT_ENOUGH_INFORMATION", "confidence": 0.0, "reason": "short explanation"}}"""


def check_claim_entailment(
    claim: dict,
    reference_text: str,
    candidate_reference_claims: list = None
) -> dict:
    """
    Check if a claim is entailed by reference text/claims.
    Strategy: deterministic structured match first, LLM fallback for text claims.
    """
    claim_text = claim.get("text", "")
    claim_value = claim.get("value")

    # --- Step 1: Deterministic structured matching ---
    if candidate_reference_claims and claim_value is not None:
        for ref_claim in candidate_reference_claims:
            if numeric_claim_match(claim, ref_claim):
                return {
                    "label": "ENTAILED",
                    "matched_claim_id": ref_claim.get("claim_id"),
                    "confidence": 0.95,
                    "reason": "Deterministic numeric match within tolerance"
                }

    # --- Step 2: Text-level entailment check via LLM ---
    if not reference_text or not reference_text.strip():
        return {
            "label": "NOT_ENOUGH_INFORMATION",
            "matched_claim_id": None,
            "confidence": 0.0,
            "reason": "No reference text available"
        }

    params = CLAIM_EVALUATION_PARAMS
    prompt = _ENTAILMENT_PROMPT.format(
        claim_text=claim_text,
        reference_text=reference_text[:3000]  # truncate for token budget
    )

    try:
        # Reuse the shared 429-resilient client if available
        eval_client = LLM_CLIENT if 'LLM_CLIENT' in dir() else None
        if eval_client is None:
            from openai import OpenAI
            workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
            _token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
            eval_client = OpenAI(api_key=_token, base_url=f"https://{workspace_url}/serving-endpoints")

        resp = eval_client.chat.completions.create(
            model=params["checker_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=params["temperature"],
            max_tokens=200
        )
        raw = resp.choices[0].message.content.strip()
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            result.setdefault("matched_claim_id", None)
            result.setdefault("confidence", 0.0)
            result.setdefault("reason", "")
            if result.get("label") not in ("ENTAILED", "CONTRADICTED", "NOT_ENOUGH_INFORMATION"):
                result["label"] = "NOT_ENOUGH_INFORMATION"
            return result
    except Exception as e:
        pass

    return {
        "label": "NOT_ENOUGH_INFORMATION",
        "matched_claim_id": None,
        "confidence": 0.0,
        "reason": f"Entailment check failed"
    }


print("✓ Claim matching & entailment loaded")

In [0]:
# =============================================================================
# CLAIM-LEVEL CORRECTNESS (Precision, Recall, F1)
# =============================================================================

def compute_claim_correctness(
    question: str,
    expected_answer: str,
    expected_claims: list,
    generated_answer: str,
    tolerance_config: dict = None
) -> dict:
    """
    RAGChecker-inspired claim-level precision, recall, F1.
    precision = generated claims entailed by ground truth / generated claims
    recall = expected claims entailed by generated answer / expected claims
    """
    result = {
        "generated_claims": [], "expected_claims": expected_claims or [],
        "precision_matches": [], "recall_matches": [],
        "correct_generated_claim_count": 0, "covered_expected_claim_count": 0,
        "generated_claim_count": 0, "expected_claim_count": len(expected_claims) if expected_claims else 0,
        "answer_precision": 0.0, "answer_recall": 0.0, "answer_f1": 0.0,
        "claim_evaluation_status": "success", "claim_evaluation_error": None,
    }

    if not expected_claims:
        result["claim_evaluation_status"] = "not_applicable"
        return result

    if not generated_answer or not generated_answer.strip():
        return result  # Empty answer = P=R=F1=0

    # Extract generated claims
    gen_claims = extract_atomic_claims(question, generated_answer, source_type="generated")
    if gen_claims and any(c.get("_evaluator_error") for c in gen_claims):
        result["claim_evaluation_status"] = "evaluator_error"
        result["claim_evaluation_error"] = gen_claims[0].get("error", "unknown")
        return result

    result["generated_claims"] = gen_claims
    result["generated_claim_count"] = len(gen_claims)
    if not gen_claims:
        return result

    # PRECISION: generated claims checked against expected answer + claims
    matched_expected_ids = set()
    precision_matches = []
    for gc in gen_claims:
        match_result = check_claim_entailment(gc, expected_answer, expected_claims)
        is_entailed = match_result.get("label") == "ENTAILED"
        matched_id = match_result.get("matched_claim_id")
        if is_entailed and matched_id and matched_id in matched_expected_ids:
            is_entailed = False  # one-to-one dedup
        if is_entailed and matched_id:
            matched_expected_ids.add(matched_id)
        precision_matches.append({"generated_claim_id": gc.get("claim_id"), "entailed": is_entailed, **match_result})
    result["precision_matches"] = precision_matches
    result["correct_generated_claim_count"] = sum(1 for m in precision_matches if m["entailed"])

    # RECALL: expected claims checked against generated answer + claims
    recall_matches = []
    for ec in expected_claims:
        match_result = check_claim_entailment(ec, generated_answer, gen_claims)
        is_covered = match_result.get("label") == "ENTAILED"
        recall_matches.append({"expected_claim_id": ec.get("claim_id"), "covered": is_covered, **match_result})
    result["recall_matches"] = recall_matches
    result["covered_expected_claim_count"] = sum(1 for m in recall_matches if m["covered"])

    # P, R, F1
    P = result["correct_generated_claim_count"] / result["generated_claim_count"] if result["generated_claim_count"] > 0 else 0.0
    R = result["covered_expected_claim_count"] / result["expected_claim_count"] if result["expected_claim_count"] > 0 else 0.0
    F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0
    result["answer_precision"] = round(P, 4)
    result["answer_recall"] = round(R, 4)
    result["answer_f1"] = round(F1, 4)
    return result


# =============================================================================
# CLAIM-LEVEL GROUNDEDNESS
# =============================================================================

def compute_claim_groundedness(
    generated_claims: list, sql_results: str = None,
    retrieved_context: str = None, direct_semantic_context: str = None
) -> dict:
    """Check generated claims against all available evidence."""
    result = {"claim_groundedness": None, "grounded_claim_count": 0, "total_factual_claims": 0, "grounding_details": []}
    factual_claims = [c for c in (generated_claims or []) if not c.get("_evaluator_error") and c.get("text")]
    if not factual_claims:
        return result
    result["total_factual_claims"] = len(factual_claims)

    evidence_parts = []
    if sql_results and str(sql_results).strip() not in ("", "None", "N/A"):
        evidence_parts.append(f"SQL RESULTS:\n{sql_results}")
    if retrieved_context and str(retrieved_context).strip() not in ("", "None", "[]"):
        evidence_parts.append(f"RETRIEVED CONTEXT:\n{retrieved_context}")
    if direct_semantic_context and str(direct_semantic_context).strip():
        evidence_parts.append(f"SEMANTIC CONTEXT:\n{direct_semantic_context[:2000]}")
    if not evidence_parts:
        return result

    evidence_text = "\n\n".join(evidence_parts)
    grounded = 0
    for claim in factual_claims:
        ent = check_claim_entailment(claim, evidence_text, None)
        is_g = ent.get("label") == "ENTAILED"
        if is_g:
            grounded += 1
        result["grounding_details"].append({"claim_id": claim.get("claim_id"), "grounded": is_g, "reason": ent.get("reason", "")})
    result["grounded_claim_count"] = grounded
    result["claim_groundedness"] = round(grounded / len(factual_claims), 4)
    return result


# =============================================================================
# UPDATED compute_all_metrics() — claim-based + legacy
# =============================================================================

def compute_all_metrics(run: dict, question: dict, mode: str) -> dict:
    """
    Full metric bundle integrating claim-level P/R/F1 and legacy heuristics.
    """
    answer = run.get("answer", "") or ""
    sql = run.get("sql_generated", "") or ""
    sql_results = run.get("sql_results", "") or ""
    expected = question.get("expected_answer", "") or ""
    expected_claims = question.get("expected_claims", [])
    tables_needed = question.get("tables_needed", [])
    answerability = question.get("answerability_label", "answerable") or "answerable"

    # Legacy metrics
    sql_status, sql_error = infer_sql_execution_status(sql_results, sql)
    evidence = run.get("evidence", sql_results)
    gnd_score_legacy, gnd_explanation = compute_groundedness_score(answer, evidence)
    hal_risk, hal_flag = compute_numeric_hallucination_risk(answer, sql_results)
    correctness_proxy = compute_answer_correctness_proxy(answer, expected)
    sql_corr = compute_sql_correctness_heuristic(sql, tables_needed, sql_status)
    sem_align = compute_semantic_alignment_heuristic(sql, mode)
    abstention = compute_abstention_correctness(answer, answerability)
    workflow_ok = run.get("success", False) and sql_status in ("success", "not_applicable") and bool(answer)
    failure = None
    if not run.get("success", False): failure = run.get("error") or "unknown error"
    elif sql_status == "failed": failure = f"SQL failed: {sql_error}"
    elif not answer: failure = "no answer generated"

    # Claim-based metrics
    claim_correctness = {"claim_evaluation_status": "not_applicable", "answer_precision": 0.0, "answer_recall": 0.0, "answer_f1": 0.0}
    claim_grounding = {"claim_groundedness": None}
    is_unanswerable = answerability.lower() not in ("answerable", "yes", "")
    if not is_unanswerable and expected_claims:
        claim_correctness = compute_claim_correctness(question.get("question", ""), expected, expected_claims, answer)
        if claim_correctness.get("generated_claims"):
            claim_grounding = compute_claim_groundedness(
                claim_correctness["generated_claims"], sql_results,
                run.get("retrieved_context", ""), run.get("direct_semantic_context", "")
            )

    # --- Derive structured KPIs from claim evaluation ---
    _tp = claim_correctness.get("correct_generated_claim_count", 0)
    _gen_count = claim_correctness.get("generated_claim_count", 0)
    _exp_count = claim_correctness.get("expected_claim_count", 0)
    _covered = claim_correctness.get("covered_expected_claim_count", 0)
    _fp = _gen_count - _tp
    _fn = _exp_count - _covered
    _grounded = claim_grounding.get("grounded_claim_count", 0)
    _total_factual = claim_grounding.get("total_factual_claims", 0)
    _hallucinated = _total_factual - _grounded if _total_factual > 0 else 0
    _halluc_rate = round(_hallucinated / _total_factual, 4) if _total_factual > 0 else None

    # --- Derive binary answer/abstention flags ---
    # Only count as abstained if the response STARTS with an explicit abstention signal.
    # Mentioning "insufficient evidence" mid-answer while still providing data is NOT abstention.
    _answer_trimmed_upper = (answer or "").strip().upper()
    # Detect abstention signal ANYWHERE in the answer (not just at the start).
    # Agents sometimes embed CANNOT_ANSWER or INSUFFICIENT_EVIDENCE mid-response
    # after a preamble like "To answer this question... CANNOT_ANSWER".
    # Using `in` instead of `startswith` ensures these cases are correctly caught.
    _contains_abstention = (
        "CANNOT_ANSWER" in _answer_trimmed_upper
        or "INSUFFICIENT_EVIDENCE" in _answer_trimmed_upper
        or _answer_trimmed_upper.startswith("ABSTAIN")
    )
    _answered = bool(answer and answer.strip() and not _contains_abstention)
    _abstained = not _answered
    _answer_correct = bool(claim_correctness.get("answer_f1", 0) > 0)

    # --- SQL binary flags ---
    _sql_attempted = bool(sql and sql.strip())
    _sql_success = (sql_status == "success")
    _sql_sem_correct = bool(sql_corr is not None and sql_corr >= 0.5)

    # --- Post-failure fabrication: answer has specific numbers despite SQL failure ---
    _post_failure_fabrication = False
    if sql_status == "failed" and _answered:
        _ans_nums = _extract_numbers(answer)
        _post_failure_fabrication = len(_ans_nums) >= 2  # 2+ unsupported numbers = fabrication

    # --- Override metrics for unanswerable questions ---
    # Correct abstention on unanswerable = perfect score (F1=1, no hallucination)
    # Wrong answer on unanswerable = failure (F1=0, full hallucination)
    if is_unanswerable:
        if _abstained:
            claim_correctness = {**claim_correctness, "answer_precision": 1.0, "answer_recall": 1.0, "answer_f1": 1.0, "claim_evaluation_status": "correct_abstention"}
            claim_grounding = {**claim_grounding, "claim_groundedness": 1.0}
            _answer_correct = True
            hal_risk = 0.0
            _halluc_rate = 0.0
            _hallucinated = 0
            _grounded = 0
            _total_factual = 0
            _post_failure_fabrication = False
        else:
            claim_correctness = {**claim_correctness, "answer_precision": 0.0, "answer_recall": 0.0, "answer_f1": 0.0, "claim_evaluation_status": "failed_abstention"}
            claim_grounding = {**claim_grounding, "claim_groundedness": 0.0}
            _answer_correct = False
            hal_risk = 1.0
            _halluc_rate = 1.0
            _post_failure_fabrication = True

    return {
        # Claim-based (primary) — thesis KPI names
        "claim_precision": claim_correctness.get("answer_precision", 0.0),
        "claim_recall": claim_correctness.get("answer_recall", 0.0),
        "claim_f1": claim_correctness.get("answer_f1", 0.0),
        "claim_true_positives": _tp,
        "claim_false_positives": _fp,
        "claim_false_negatives": _fn,
        "grounded_claims": _grounded,
        "hallucinated_claims": _hallucinated,
        "total_factual_claims": _total_factual,
        "groundedness": claim_grounding.get("claim_groundedness"),
        "hallucination_rate": _halluc_rate,
        # Binary KPIs
        "answer_correct": _answer_correct,
        "answered": _answered,
        "abstained": _abstained,
        "sql_attempted": _sql_attempted,
        "sql_execution_success": _sql_success,
        "sql_semantically_correct": _sql_sem_correct,
        "post_failure_fabrication": _post_failure_fabrication,
        # Aliases
        "answer_precision": claim_correctness.get("answer_precision", 0.0),
        "answer_recall": claim_correctness.get("answer_recall", 0.0),
        "answer_f1": claim_correctness.get("answer_f1", 0.0),
        "generated_claim_count": _gen_count,
        "expected_claim_count": _exp_count,
        "correct_generated_claim_count": _tp,
        "covered_expected_claim_count": _covered,
        "claim_groundedness": claim_grounding.get("claim_groundedness"),
        "claim_evaluation_status": claim_correctness.get("claim_evaluation_status", "not_applicable"),
        "claim_evaluation_error": claim_correctness.get("claim_evaluation_error"),
        # Supporting metrics
        "answer_correctness_proxy": correctness_proxy,
        "groundedness_score": gnd_score_legacy,
        "grounding_explanation": gnd_explanation,
        "numeric_hallucination_risk": hal_risk,
        "hallucination_flag": hal_flag,
        "abstention_correctness": abstention,
        "semantic_alignment_score": sem_align,
        # SQL
        "selected_tables": extract_tables_from_sql(sql),
        "generated_sql": sql or None,
        "sql_execution_status": sql_status,
        "sql_error_message": sql_error,
        "sql_correctness_score": sql_corr,
        "sql_result_summary": str(sql_results)[:500] if sql_results else None,
        # Workflow
        "workflow_success": workflow_ok,
        "failure_reason": failure,
    }


print("✓ Claim correctness, groundedness & compute_all_metrics() v3 loaded")
print("  Thesis KPIs: claim_precision, claim_recall, claim_f1, claim_TP/FP/FN")
print("  Thesis KPIs: groundedness, hallucination_rate (claim-based)")
print("  Thesis KPIs: answer_correct, answered, abstained, sql_execution_success")
print("  Thesis KPIs: post_failure_fabrication, sql_semantically_correct")


In [0]:
# =============================================================================
# VALIDATION TESTS — Claim-Level Evaluation
# Run only when explicitly requested (guarded by _RUN_CLAIM_TESTS flag)
# =============================================================================

_RUN_CLAIM_TESTS = False  # Set True to run validation suite

def run_claim_evaluation_tests():
    """Execute validation test suite for claim-level evaluation."""
    import traceback
    results = []

    def _test(name, fn):
        try:
            passed, detail = fn()
            results.append({"test": name, "passed": passed, "detail": detail})
        except Exception as e:
            results.append({"test": name, "passed": False, "detail": f"ERROR: {e}\n{traceback.format_exc()[:200]}"})

    # --- Test 1: Exact numeric match ---
    def t1_exact_numeric():
        gc = {"claim_id": "g1", "text": "324 certificates", "entity": None, "metric": "certificate count", "value": 324, "unit": None, "period": "february 2026", "qualifiers": {}, "required": True}
        ec = {"claim_id": "e1", "text": "324 certificates", "entity": None, "metric": "certificate count", "value": 324, "unit": None, "period": "february 2026", "qualifiers": {}, "required": True}
        return numeric_claim_match(gc, ec), "Count exact match"
    _test("Exact numeric count match", t1_exact_numeric)

    # --- Test 2: Percentage within tolerance (0.5pp) ---
    def t2_pct_within():
        gc = {"claim_id": "g1", "text": "32%", "value": 32.0, "unit": "%", "metric": "rate", "entity": None, "period": None, "qualifiers": {}, "required": True}
        ec = {"claim_id": "e1", "text": "31.9%", "value": 31.9, "unit": "%", "metric": "rate", "entity": None, "period": None, "qualifiers": {}, "required": True}
        return numeric_claim_match(gc, ec), "32.0 vs 31.9 within 0.5pp"
    _test("Percentage within tolerance", t2_pct_within)

    # --- Test 3: Percentage outside tolerance ---
    def t3_pct_outside():
        gc = {"claim_id": "g1", "text": "35%", "value": 35.0, "unit": "%", "metric": "rate", "entity": None, "period": None, "qualifiers": {}, "required": True}
        ec = {"claim_id": "e1", "text": "31.9%", "value": 31.9, "unit": "%", "metric": "rate", "entity": None, "period": None, "qualifiers": {}, "required": True}
        return not numeric_claim_match(gc, ec), "35.0 vs 31.9 exceeds 0.5pp"
    _test("Percentage outside tolerance", t3_pct_outside)

    # --- Test 4: Wrong entity, correct number ---
    def t4_wrong_entity():
        gc = {"claim_id": "g1", "text": "company 831 has 240", "entity": "company_id2=831", "metric": "certificate count", "value": 240, "unit": None, "period": "2026", "qualifiers": {}, "required": True}
        ec = {"claim_id": "e1", "text": "company 1039 has 240", "entity": "company_id2=1039", "metric": "certificate count", "value": 240, "unit": None, "period": "2026", "qualifiers": {}, "required": True}
        return not numeric_claim_match(gc, ec), "Same number but different entity"
    _test("Wrong entity with correct number", t4_wrong_entity)

    # --- Test 5: Wrong period with correct number ---
    def t5_wrong_period():
        gc = {"claim_id": "g1", "text": "53 users in feb", "entity": None, "metric": "distinct users", "value": 53, "unit": None, "period": "february 2026", "qualifiers": {}, "required": True}
        ec = {"claim_id": "e1", "text": "53 users in jan", "entity": None, "metric": "distinct users", "value": 53, "unit": None, "period": "january 2026", "qualifiers": {}, "required": True}
        return not numeric_claim_match(gc, ec), "Same count but different period"
    _test("Wrong period with correct number", t5_wrong_period)

    # --- Test 6: EU decimal format ---
    def t6_eu_format():
        val = normalise_numeric_string("1.234,56")
        return val == 1234.56, f"EU '1.234,56' = {val}"
    _test("European decimal format", t6_eu_format)

    # --- Test 7: K/M suffixes ---
    def t7_suffix():
        val_k = normalise_numeric_string("1.5k")
        val_m = normalise_numeric_string("2.3M")
        return val_k == 1500.0 and val_m == 2_300_000.0, f"1.5k={val_k}, 2.3M={val_m}"
    _test("Numeric suffixes (k/m)", t7_suffix)

    # --- Test 8: Empty answer for answerable question ---
    def t8_empty_answer():
        ec = [{"claim_id": "e1", "text": "324 certs", "value": 324, "entity": None, "metric": "count", "unit": None, "period": None, "qualifiers": {}, "required": True}]
        r = compute_claim_correctness("How many?", "324 certs", ec, "")
        return r["answer_f1"] == 0.0 and r["claim_evaluation_status"] == "success", f"F1={r['answer_f1']}"
    _test("Empty answer = zero scores", t8_empty_answer)

    # --- Test 9: Unanswerable question ---
    def t9_unanswerable():
        r = compute_claim_correctness("Impossible q", "No data", [], "Some answer")
        return r["claim_evaluation_status"] == "not_applicable", f"status={r['claim_evaluation_status']}"
    _test("Unanswerable = not_applicable", t9_unanswerable)

    # --- Test 10: Normalisation ---
    def t10_normalise():
        n1 = normalise_text("  Hello   WORLD  ")
        n2 = normalise_numeric_string("$1,234.56")
        n3 = normalise_numeric_string("68%")
        return n1 == "hello world" and n2 == 1234.56 and n3 == 68.0, f"{n1}, {n2}, {n3}"
    _test("Text/numeric normalisation", t10_normalise)

    # --- Print results ---
    passed = sum(1 for r in results if r["passed"])
    total = len(results)
    print(f"\n{'='*60}")
    print(f"CLAIM EVALUATION TESTS: {passed}/{total} passed")
    print(f"{'='*60}")
    for r in results:
        icon = "✓" if r["passed"] else "✗"
        print(f"  {icon} {r['test']}: {r['detail']}")
    print()
    return results


# Auto-run if flag set
if _RUN_CLAIM_TESTS:
    run_claim_evaluation_tests()